# Agentic AI Sycophancy: GitHub Data Collection

This notebook collects GitHub issue-thread data for a study on sycophancy in agentic AI systems. It adapts the data collection workflow from the TRACE framework study. Coding of the collected threads is handled separately.

The notebook:
1. verifies and canonicalises the 30 target repositories against the GitHub API,
2. searches public GitHub issues created within a fixed date window, using a typed sycophancy search lexicon,
3. retrieves all results per query, splitting date windows recursively where GitHub's 1,000-result cap is exceeded, and logs every executed query,
4. removes duplicate issues retrieved by multiple search terms or adjacent windows (recording all matching terms and term types),
5. retrieves public issue comments for all unique candidate issues, and
6. constructs one structured thread text per issue, with author, role, and timestamp markers, saved as CSV and Excel files.

The resulting dataset is a keyword-enriched candidate corpus, suitable for taxonomy development and qualitative analysis. It is not a prevalence sample: claims about how often sycophancy occurs on GitHub would require an independent background sample as a denominator.

API keys are not included. Add `GITHUB_TOKEN` through Google Colab Secrets before running the notebook.


In [ ]:
!pip install -q requests pandas tqdm openpyxl

In [ ]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    print("No GitHub token found. Public API access may still work, but rate limits will be much lower.")
else:
    print("GitHub token loaded.")

In [ ]:
HEADERS = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"


In [ ]:
# 30-repository balanced corpus: five repositories in each of six functional categories.
# Categories follow the study design; repo paths are canonicalised in the next cell.

REPO_CATEGORIES = {
    # 1. General-purpose agent SDKs and orchestration frameworks
    "microsoft/agent-framework": "Agent SDKs and orchestration",
    "microsoft/semantic-kernel": "Agent SDKs and orchestration",
    "langchain-ai/langgraph": "Agent SDKs and orchestration",
    "openai/openai-agents-python": "Agent SDKs and orchestration",
    "google/adk-python": "Agent SDKs and orchestration",

    # 2. Multi-agent collaboration frameworks
    "microsoft/autogen": "Multi-agent collaboration",
    "crewAIInc/crewAI": "Multi-agent collaboration",
    "camel-ai/camel": "Multi-agent collaboration",
    "FoundationAgents/MetaGPT": "Multi-agent collaboration",
    "OpenBMB/ChatDev": "Multi-agent collaboration",

    # 3. Coding and software-engineering agents
    "OpenHands/OpenHands": "Coding and software-engineering agents",
    "cline/cline": "Coding and software-engineering agents",
    "openai/codex": "Coding and software-engineering agents",
    "google-gemini/gemini-cli": "Coding and software-engineering agents",
    "anthropics/claude-code": "Coding and software-engineering agents",

    # 4. Browser, research, and general action agents
    "browser-use/browser-use": "Browser, research, and action agents",
    "FoundationAgents/OpenManus": "Browser, research, and action agents",
    "assafelovic/gpt-researcher": "Browser, research, and action agents",
    "langchain-ai/open_deep_research": "Browser, research, and action agents",
    "SamuelSchmidgall/AgentLaboratory": "Browser, research, and action agents",

    # 5. Memory, personalisation, and stateful agent infrastructure
    "mem0ai/mem0": "Memory and personalisation infrastructure",
    "letta-ai/letta": "Memory and personalisation infrastructure",
    "getzep/graphiti": "Memory and personalisation infrastructure",
    "langchain-ai/langmem": "Memory and personalisation infrastructure",
    "langgenius/dify": "Memory and personalisation infrastructure",

    # 6. Evaluation, observability, and control systems
    "promptfoo/promptfoo": "Evaluation, observability, and control",
    "confident-ai/deepeval": "Evaluation, observability, and control",
    "langfuse/langfuse": "Evaluation, observability, and control",
    "Arize-ai/phoenix": "Evaluation, observability, and control",
    "AgentOps-AI/agentops": "Evaluation, observability, and control",
}

REPOS = list(REPO_CATEGORIES.keys())

# Search terms derived deductively from the sycophancy construct.
# Multi-word terms are searched as quoted phrases, which reduces noise.
SYCOPHANCY_TERMS = {
    "Agreement and Opinion Conformity": [
        "sycophancy",
        "sycophantic",
        "too agreeable",
        "overly agreeable",
        "always agrees",
        "agrees with everything",
        "yes-man",
        "people pleasing",
        "blind agreement",
        "user is always right",
        "uncritically agrees",
        "accepts false premise",
        "accepts a false premise",
        "agrees with an incorrect premise",
        "mirrors the user's opinion",
        "echoes the user's view",
        "confirms the user's belief",
        "validates an incorrect belief"
    ],
    "Capitulation and Answer Flipping": [
        "backs down",
        "changes its answer",
        "changes answer after pushback",
        "changes answer when challenged",
        "reverses a correct answer",
        "backs down when challenged",
        "abandons the correct answer",
        "second-guesses a correct answer",
        "capitulate",
        "flip-flop",
        "flip flops",
        "over-apologetic",
        "overly apologetic",
        "apologizes too much",
        "apologizes and changes"
    ],
    "Flattery and Excessive Validation": [
        "flattery",
        "you're absolutely right",
        "excessive praise",
        "excessively praises the user",
        "praises the user",
        "excessive compliments",
        "glazing",
        "excessive validation",
        "uncritical validation",
        "validates a bad idea",
        "over-validation",
        "overvalidates",
        "over-validates"
    ],
    "Confirmation-Biased Evidence": [
        "confirmation bias",
        "cherry-pick",
        "what you want to hear",
        "echo chamber",
        "selective evidence",
        "selective retrieval",
        "ignores contradictory evidence",
        "seeks supporting evidence",
        "predetermined conclusion",
        "slanted summary",
        "confirms the user's conclusion",
        "researches only one side",
        "one-sided summary",
        "one-sided evidence",
        "biased summary"
    ],
    "Agentic Action Sycophancy": [
        "fails to challenge",
        "does not push back",
        "blindly follows",
        "blindly agrees",
        "uncritical compliance",
        "accepts a bad plan",
        "ignores constraints",
        "changes plan to please"
    ],
    "Inter-Agent Deference and Consensus": [
        "groupthink",
        "premature consensus",
        "disagreement collapse",
        "false consensus",
        "consensus collapse",
        "agent herding",
        "rubber-stamp",
        "rubber stamp",
        "agents converge too quickly",
        "agrees with the supervisor",
        "follows the supervisor blindly",
        "uncritically accepts the planner",
        "accepts the critic's answer",
        "defers to"
    ],
    "Evaluator and Judge Bias": [
        "judge bias",
        "inflated score",
        "grade inflation",
        "self-preference",
        "verbosity bias",
        "position bias",
        "reward hacking"
    ],
    "Personalisation and Memory Sycophancy": [
        "pandering",
        "personalization bias",
        "personalisation bias",
        "over-personalization",
        "over-personalisation",
        "tells the user what they want",
        "memory-induced sycophancy",
        "uses preference as fact",
        "preference overrides evidence",
        "overweights user preference",
        "reinforces a misconception",
        "remembered belief as fact",
        "mirrors the user",
        "reinforces beliefs",
        "stale preference",
        "inferred preference"
    ]
}

n_terms = sum(len(v) for v in SYCOPHANCY_TERMS.values())
print(f"{len(REPOS)} repositories, {n_terms} search terms, "
      f"up to {len(REPOS) * n_terms} repo-term queries.")


In [ ]:
# Verify and canonicalise repository names before searching.
# GitHub's search API can silently return zero results for renamed or moved
# repositories, so every path is resolved to its current canonical location here.

import requests
import time

VERIFIED_REPO_CATEGORIES = {}
problems = []

for repo, category in REPO_CATEGORIES.items():
    response = requests.get(f"https://api.github.com/repos/{repo}", headers=HEADERS)

    if response.status_code == 200:
        canonical = response.json().get("full_name", repo)
        if canonical != repo:
            print(f"Renamed: {repo} -> {canonical}")
        VERIFIED_REPO_CATEGORIES[canonical] = category
    else:
        problems.append((repo, response.status_code))
        print(f"PROBLEM: {repo} returned HTTP {response.status_code}")

    time.sleep(0.5)

REPOS = list(VERIFIED_REPO_CATEGORIES.keys())
REPO_CATEGORIES = VERIFIED_REPO_CATEGORIES

print(f"\n{len(REPOS)} repositories verified.")
if problems:
    print(f"{len(problems)} repositories could not be resolved and will be skipped: {problems}")
    print("Check these paths manually on github.com before proceeding.")


In [ ]:
# Reproducibility settings

from datetime import date

# Fixed collection window. Only issues CREATED within this window are collected,
# which makes the corpus reproducible: rerunning the notebook later returns the
# same issue population (up to post-hoc deletions and edits on GitHub).
START_DATE = date(2023, 1, 1)
END_DATE = date(2026, 6, 30)

# Search API pacing. The authenticated GitHub search limit is 30 requests per
# minute, so a 2-second sleep between search requests avoids most rate limiting.
SEARCH_SLEEP_SECONDS = 2

# GitHub search returns at most 1,000 results per query (10 pages of 100).
# Queries exceeding the cap are split recursively into smaller date windows.
PER_PAGE = 100
MAX_PAGES_PER_QUERY = 10
API_RESULT_CAP = 1000

# Output filenames
CANDIDATE_CSV = "github_sycophancy_candidate_issues.csv"
CANDIDATE_EXCEL = "github_sycophancy_candidate_issues_clean.xlsx"

COMMENTS_CSV = "github_sycophancy_issue_comments.csv"
COMMENTS_EXCEL = "github_sycophancy_issue_comments.xlsx"

THREADS_CSV = "github_sycophancy_issue_threads.csv"
THREADS_EXCEL = "github_sycophancy_issue_threads.xlsx"

SEARCH_LOG_CSV = "github_sycophancy_search_log.csv"


In [ ]:
# Optional: corpus size estimate (dry run)
#
# Set RUN_DRY_RUN = True to query only the total_count for every repo-term pair
# (one API request each, no pagination, no issue bodies). This reports the exact
# number of raw hits per query before deduplication, at a fraction of the cost
# of full collection, and saves the counts to a CSV for inspection.
# At the standard pacing this takes roughly 1.5 to 2 hours for 3,180 queries.

RUN_DRY_RUN = False

if RUN_DRY_RUN:
    import time
    import requests
    import pandas as pd
    from tqdm import tqdm

    count_rows = []

    for repo in tqdm(REPOS, desc="Repositories"):
        for dimension, terms in SYCOPHANCY_TERMS.items():
            for term in terms:
                query = (
                    f'repo:{repo} is:issue "{term}" '
                    f'created:{START_DATE.isoformat()}..{END_DATE.isoformat()}'
                )
                response = requests.get(
                    "https://api.github.com/search/issues",
                    headers=HEADERS,
                    params={"q": query, "per_page": 1}
                )
                if response.status_code == 200:
                    total = response.json().get("total_count")
                else:
                    total = None
                    if response.status_code in [403, 429]:
                        time.sleep(60)

                count_rows.append({
                    "repo": repo,
                    "dimension": dimension,
                    "search_term": term,
                    "total_count": total
                })
                time.sleep(SEARCH_SLEEP_SECONDS)

    counts_df = pd.DataFrame(count_rows)
    counts_df.to_csv("github_sycophancy_dry_run_counts.csv",
                     index=False, encoding="utf-8-sig")

    print(f"Total raw hits (before deduplication): {counts_df['total_count'].sum():,.0f}")
    print("\nTop 20 repo-term pairs by hit count:")
    display(counts_df.sort_values("total_count", ascending=False).head(20))
    print("\nHits by search term (summed over repositories):")
    display(counts_df.groupby("search_term")["total_count"].sum()
            .sort_values(ascending=False).head(30))


In [ ]:
import math
import pandas as pd
from datetime import timedelta
from tqdm import tqdm


def github_get(url, params=None, max_retries=5):
    for attempt in range(max_retries):
        response = requests.get(url, headers=HEADERS, params=params)

        if response.status_code == 200:
            return response.json()

        if response.status_code in [403, 429]:
            reset_time = response.headers.get("x-ratelimit-reset")
            remaining = response.headers.get("x-ratelimit-remaining")

            if remaining == "0" and reset_time:
                sleep_for = max(int(reset_time) - int(time.time()) + 5, 10)
                print(f"Rate limit reached. Sleeping for {sleep_for} seconds.")
                time.sleep(sleep_for)
            else:
                wait = 60 * (attempt + 1)
                print(f"Secondary rate limit. Sleeping for {wait} seconds.")
                time.sleep(wait)
            continue

        print(f"Request failed: {response.status_code}")
        print(response.text[:300])
        return None

    return None


SEARCH_URL = "https://api.github.com/search/issues"


def item_to_row(item, repo, term, dimension):
    return {
        "syco_search_dimension": dimension,
        "search_term": term,
        "repo": repo,
        "repo_category": REPO_CATEGORIES.get(repo, ""),
        "issue_number": item.get("number"),
        "title": item.get("title"),
        "body": item.get("body"),
        "issue_author": item.get("user", {}).get("login") if item.get("user") else None,
        "issue_author_association": item.get("author_association"),
        "state": item.get("state"),
        "created_at": item.get("created_at"),
        "updated_at": item.get("updated_at"),
        "closed_at": item.get("closed_at"),
        "comments_count": item.get("comments"),
        "labels": "; ".join([label["name"] for label in item.get("labels", [])]),
        "html_url": item.get("html_url"),
        "comments_url": item.get("comments_url")
    }


def search_issues_window(repo, term, dimension,
                         window_start, window_end, search_log):
    """
    Exhaustively retrieve all issues matching a repo-term query within a date
    window. If the window exceeds GitHub's 1,000-result cap, it is split
    recursively into two smaller windows. Every executed query is logged with
    its total_count, retrieved count, and truncation status.
    """
    rows = []
    query = (
        f'repo:{repo} is:issue "{term}" '
        f'created:{window_start.isoformat()}..{window_end.isoformat()}'
    )

    params = {"q": query, "sort": "created", "order": "asc",
              "per_page": PER_PAGE, "page": 1}
    data = github_get(SEARCH_URL, params=params)
    time.sleep(SEARCH_SLEEP_SECONDS)

    if not data or "items" not in data:
        search_log.append({
            "repo": repo, "search_term": term,
            "dimension": dimension, "window_start": window_start.isoformat(),
            "window_end": window_end.isoformat(), "total_count": None,
            "retrieved": 0, "action": "request_failed", "truncated": True
        })
        return rows

    total_count = data.get("total_count", 0)

    # Over the API cap: split the window in half and recurse (if splittable).
    if total_count > API_RESULT_CAP and window_start < window_end:
        mid = window_start + (window_end - window_start) / 2
        search_log.append({
            "repo": repo, "search_term": term,
            "dimension": dimension, "window_start": window_start.isoformat(),
            "window_end": window_end.isoformat(), "total_count": total_count,
            "retrieved": 0, "action": "split", "truncated": False
        })
        rows.extend(search_issues_window(
            repo, term, dimension, window_start, mid, search_log))
        rows.extend(search_issues_window(
            repo, term, dimension, mid + timedelta(days=1), window_end, search_log))
        return rows

    for item in data["items"]:
        rows.append(item_to_row(item, repo, term, dimension))

    pages_needed = min(math.ceil(total_count / PER_PAGE), MAX_PAGES_PER_QUERY)

    for page in range(2, pages_needed + 1):
        params["page"] = page
        data = github_get(SEARCH_URL, params=params)
        time.sleep(SEARCH_SLEEP_SECONDS)

        if not data or "items" not in data or len(data["items"]) == 0:
            break

        for item in data["items"]:
            rows.append(item_to_row(item, repo, term, dimension))

    # Truncated only in the unsplittable case: a single-day window over the cap.
    truncated = total_count > API_RESULT_CAP

    search_log.append({
        "repo": repo, "search_term": term,
        "dimension": dimension, "window_start": window_start.isoformat(),
        "window_end": window_end.isoformat(), "total_count": total_count,
        "retrieved": len(rows), "action": "collected", "truncated": truncated
    })

    return rows


all_rows = []
search_log = []

for repo in tqdm(REPOS, desc="Repositories"):
    for dimension, terms in SYCOPHANCY_TERMS.items():
        for term in tqdm(terms, desc=repo, leave=False):
            all_rows.extend(search_issues_window(
                repo, term, dimension,
                START_DATE, END_DATE, search_log))

search_log_df = pd.DataFrame(search_log)
search_log_df.to_csv(SEARCH_LOG_CSV, index=False, encoding="utf-8-sig")

truncated_queries = search_log_df[search_log_df["truncated"] == True]
print(f"Executed queries logged: {len(search_log_df)}")
print(f"Truncated or failed queries: {len(truncated_queries)}")

issues_df = pd.DataFrame(all_rows)

# Duplicates can occur across search terms and across adjacent date windows.
# Before removing them, record every search term, facet, and term type that
# retrieved each issue, so multi-term hits are not lost.
term_map = (
    issues_df
    .groupby(["repo", "issue_number"])
    .agg(
        all_search_terms=("search_term", lambda s: "; ".join(sorted(set(s)))),
        all_search_dimensions=("syco_search_dimension", lambda s: "; ".join(sorted(set(s))))
    )
    .reset_index()
)

issues_df = issues_df.drop_duplicates(subset=["repo", "issue_number"])
issues_df = issues_df.merge(term_map, on=["repo", "issue_number"], how="left")

issues_df.to_csv(CANDIDATE_CSV, index=False, encoding="utf-8-sig")

print(f"Collected {len(issues_df)} unique candidate issues.")
issues_df.head()


In [ ]:
import re

# If issues_df is still in memory, use it.
# If not, reload from the CSV that was already saved.
try:
    issues_df
    print("Using existing issues_df in memory.")
except NameError:
    issues_df = pd.read_csv(CANDIDATE_CSV)
    print("Reloaded issues_df from CSV.")

# Remove illegal Excel characters
ILLEGAL_CHARACTERS_RE = re.compile(r"[\000-\010]|[\013-\014]|[\016-\037]")

def clean_excel_text(value):
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub("", value)
    return value

issues_df_clean = issues_df.applymap(clean_excel_text)

# Save cleaned Excel version
issues_df_clean.to_excel(CANDIDATE_EXCEL, index=False)

print(f"Saved cleaned Excel file with {len(issues_df_clean)} unique candidate issues.")


In [ ]:
from google.colab import files

files.download(CANDIDATE_EXCEL)
files.download(CANDIDATE_CSV)
files.download(SEARCH_LOG_CSV)


In [ ]:
# Load the cleaned candidate issue dataset
issues_df = pd.read_excel(CANDIDATE_EXCEL)

# Make sure comments_count is numeric
issues_df["comments_count"] = pd.to_numeric(
    issues_df["comments_count"], errors="coerce"
).fillna(0).astype(int)

# All unique candidate issues proceed to comment retrieval and coding.
# No comment-count filter or per-dimension sampling is applied.
issues_for_coding = issues_df.copy().reset_index(drop=True)

print("Candidate issues by repository category:")
display(issues_for_coding["repo_category"].value_counts())

print("\nCandidate issues by sycophancy search dimension (first-retrieved):")
display(issues_for_coding["syco_search_dimension"].value_counts())

print(f"Prepared {len(issues_for_coding)} unique candidate issues for comment retrieval.")


In [ ]:
def get_issue_comments(comments_url):
    comments = []
    page = 1

    while True:
        params = {
            "per_page": 100,
            "page": page
        }

        data = github_get(comments_url, params=params)

        if not data or len(data) == 0:
            break

        for comment in data:
            comments.append({
                "comment_id": comment.get("id"),
                "comment_author": comment.get("user", {}).get("login") if comment.get("user") else None,
                "comment_author_association": comment.get("author_association"),
                "comment_created_at": comment.get("created_at"),
                "comment_updated_at": comment.get("updated_at"),
                "comment_body": comment.get("body"),
                "comment_url": comment.get("html_url")
            })

        page += 1
        time.sleep(1)

    return comments


# Comment retrieval is the longest stage. Progress is checkpointed to
# COMMENTS_CSV every CHECKPOINT_EVERY processed issues, and already-retrieved
# issues are skipped on rerun, so an interrupted session resumes without loss.
CHECKPOINT_EVERY = 200

import os

if os.path.exists(COMMENTS_CSV):
    existing_comments_df = pd.read_csv(COMMENTS_CSV)
    all_comments = existing_comments_df.to_dict("records")
    done_issues = set(zip(existing_comments_df["repo"],
                          existing_comments_df["issue_number"]))
    print(f"Resuming: comments for {len(done_issues)} issues already retrieved.")
else:
    all_comments = []
    done_issues = set()

processed_since_checkpoint = 0

for _, row in tqdm(issues_for_coding.iterrows(), total=len(issues_for_coding)):
    comments_url = row["comments_url"]

    # Issues with no comments have nothing to retrieve
    if pd.isna(comments_url) or row["comments_count"] == 0:
        continue

    if (row["repo"], row["issue_number"]) in done_issues:
        continue

    comments = get_issue_comments(comments_url)

    for comment in comments:
        comment.update({
            "repo": row["repo"],
            "repo_category": row["repo_category"],
            "issue_number": row["issue_number"],
            "issue_title": row["title"],
            "issue_url": row["html_url"],
            "syco_search_dimension": row["syco_search_dimension"],
            "search_term": row["search_term"]
        })

        all_comments.append(comment)

    processed_since_checkpoint += 1
    if processed_since_checkpoint >= CHECKPOINT_EVERY:
        pd.DataFrame(all_comments).to_csv(COMMENTS_CSV, index=False, encoding="utf-8-sig")
        processed_since_checkpoint = 0

comments_df = pd.DataFrame(all_comments)

comments_df.to_csv(COMMENTS_CSV, index=False, encoding="utf-8-sig")

print(f"Collected {len(comments_df)} comments from {len(issues_for_coding)} candidate issues.")
comments_df.head()


In [ ]:
thread_rows = []

for _, issue in issues_for_coding.iterrows():
    repo = issue["repo"]
    issue_number = issue["issue_number"]

    # Structured thread text with speaker and timestamp markers, in
    # chronological order. The raw comments file remains the authoritative
    # source for comment-level metadata.
    author = issue.get("issue_author", "") or "unknown"
    association = issue.get("issue_author_association", "") or "NONE"
    created = issue.get("created_at", "") or ""

    parts = [
        f"[ISSUE | {author} ({association}) | {created}]",
        f"TITLE: {issue.get('title', '')}",
        "",
        "BODY:",
        str(issue.get("body", "") or "")
    ]

    if len(comments_df) > 0:
        issue_comments = comments_df[
            (comments_df["repo"] == repo) &
            (comments_df["issue_number"] == issue_number)
        ].sort_values("comment_created_at")

        for j, (_, comment) in enumerate(issue_comments.iterrows(), start=1):
            c_author = comment.get("comment_author", "") or "unknown"
            c_association = comment.get("comment_author_association", "") or "NONE"
            c_created = comment.get("comment_created_at", "") or ""
            c_body = str(comment.get("comment_body", "") or "")

            parts.append("")
            parts.append(f"[COMMENT {j} | {c_author} ({c_association}) | {c_created}]")
            parts.append(c_body)

    full_text = "\n".join(parts).strip()

    thread_rows.append({
        "repo": repo,
        "repo_category": issue.get("repo_category", ""),
        "issue_number": issue_number,
        "title": issue.get("title", ""),
        "issue_author": issue.get("issue_author", ""),
        "issue_author_association": issue.get("issue_author_association", ""),
        "state": issue.get("state", ""),
        "created_at": issue.get("created_at", ""),
        "updated_at": issue.get("updated_at", ""),
        "closed_at": issue.get("closed_at", ""),
        "comments_count": issue.get("comments_count", 0),
        "labels": issue.get("labels", ""),
        "html_url": issue.get("html_url", ""),
        "syco_search_dimension": issue.get("syco_search_dimension", ""),
        "search_term": issue.get("search_term", ""),
        "all_search_terms": issue.get("all_search_terms", ""),
        "all_search_dimensions": issue.get("all_search_dimensions", ""),
        "full_text": full_text
    })

threads_df = pd.DataFrame(thread_rows)

threads_df.to_csv(THREADS_CSV, index=False, encoding="utf-8-sig")

print(f"Prepared {len(threads_df)} issue threads.")
threads_df.head()


In [ ]:
threads_df_clean = threads_df.applymap(clean_excel_text)
comments_df_clean = comments_df.applymap(clean_excel_text)

threads_df_clean.to_excel(THREADS_EXCEL, index=False)
comments_df_clean.to_excel(COMMENTS_EXCEL, index=False)

print("Saved cleaned Excel files.")


In [ ]:
files.download(COMMENTS_EXCEL)
files.download(THREADS_EXCEL)


In [ ]:
# Final check and collection summary
threads_check = pd.read_excel(THREADS_EXCEL)

print(f"Threads prepared for coding: {len(threads_check)}")

print("\nThreads by repository category:")
display(threads_check["repo_category"].value_counts())

print("\nThreads by repository:")
display(threads_check["repo"].value_counts())


In [ ]:
print("Final reproducibility settings")
print(f"Repositories: {len(REPOS)}")
print(f"Search terms: {sum(len(v) for v in SYCOPHANCY_TERMS.values())}")
print(f"Collection window (issue created): {START_DATE.isoformat()} to {END_DATE.isoformat()}")
print(f"Search sleep seconds: {SEARCH_SLEEP_SECONDS}")
print(f"Per-query result cap handling: recursive date-window splitting at {API_RESULT_CAP} results")
